# Baseline Models on Washington RGB-D Object 51-Category

**Four fusion baselines using ResNet18 (0.75x width):**
1. RGB-only (single stream, 3-channel input)
2. Depth-only (single stream, 1-channel input)
3. Early fusion (RGB+Depth concatenated = 4-channel input)
4. Late fusion (two backbones, features concatenated before classifier)

Each baseline gets its own config dict (aug, LR, WD, dropout, label smoothing, grad clip, epochs, scheduler, early-stopping patience) so HPO-selected hyperparameters can be applied independently per model. Mean class accuracy (MCA) is the primary metric.

**Early stopping:** Each run tracks the best val_mca and snapshots weights when it improves. If val_mca does not improve for `early_stopping_patience` consecutive epochs, training stops early and the best weights are restored before the test eval. If the full schedule completes without triggering, the final-epoch weights are used for the test eval.

## 1. Environment Setup & GPU Verification

In [ ]:
# Check GPU availability and specs
import torch
import subprocess

print("=" * 60)
print("GPU VERIFICATION")
print("=" * 60)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

    gpu_name = torch.cuda.get_device_name(0)
    if 'A100' in gpu_name:
        print("\nA100 GPU detected - optimal for training")
    elif 'V100' in gpu_name:
        print("\nV100 GPU detected - good for training (slower than A100)")
    elif 'T4' in gpu_name:
        print("\nT4 GPU detected - will be slower, consider upgrading to A100")
    else:
        print(f"\nGPU: {gpu_name}")
else:
    print("\nNO GPU DETECTED!")
    print("Enable GPU: Runtime -> Change runtime type -> Hardware accelerator: GPU")
    raise RuntimeError("GPU is required for training")

print("\n" + "=" * 60)

In [ ]:
# Detailed GPU info
!nvidia-smi

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')

print("\nGoogle Drive mounted successfully!")
print(f"\nShared drive contents:")
!ls -la /content/drive/Shareddrives/MSNN-Capstone/ | head -20

## 3. Clone Repository to Local Disk (Fast I/O)

**Important:** We clone to `/content/` (local SSD) instead of Drive for 10-20x faster I/O

In [ ]:
import os
from pathlib import Path

PROJECT_NAME = "MSNN-Capstone"
GITHUB_REPO = "https://github.com/clingergab/MSNN-Capstone.git"
LOCAL_REPO_PATH = f"/content/{PROJECT_NAME}"

print("=" * 60)
print("REPOSITORY SETUP")
print("=" * 60)

os.chdir('/content')

if Path(LOCAL_REPO_PATH).exists() and Path(f"{LOCAL_REPO_PATH}/.git").exists():
    print(f"Repo already exists: {LOCAL_REPO_PATH}")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
else:
    if Path(LOCAL_REPO_PATH).exists():
        !rm -rf {LOCAL_REPO_PATH}
    print(f"Cloning from {GITHUB_REPO}...")
    !git clone {GITHUB_REPO} {LOCAL_REPO_PATH}
    if not Path(LOCAL_REPO_PATH).exists():
        raise RuntimeError(f"Failed to clone repository")
    os.chdir(LOCAL_REPO_PATH)

print(f"\nWorking directory: {os.getcwd()}")
!ls -la {LOCAL_REPO_PATH}
print("\n" + "=" * 60)

## 4. Install Dependencies

In [ ]:
print("Installing dependencies...")

!pip install -q h5py tqdm matplotlib seaborn kornia thop

import h5py
import tqdm
import matplotlib
import seaborn
import kornia
import thop

print("All dependencies installed!")
print(f"   h5py: {h5py.__version__}")
print(f"   matplotlib: {matplotlib.__version__}")
print(f"   kornia: {kornia.__version__}")
print(f"   thop: {thop.__version__}")

## 5. Copy Washington RGB-D Dataset to Local Disk

**Performance Note:** Local disk (`/dev/shm`) is ~10-20x faster than Drive!

**Dataset:** Washington RGB-D Object 51-category preprocessed (train + val + test splits, paired `.pt` tensors)

In [ ]:
from pathlib import Path
import os

DRIVE_DATASET_TAR = "/content/drive/Shareddrives/MSNN-Capstone/data/washington_rgbd_256.tar.gz"
LOCAL_DATASET_PATH = "/dev/shm/washington_rgbd_256"

print("=" * 60)
print("WASHINGTON RGB-D OBJECT DATASET SETUP (51 CATEGORIES)")
print("=" * 60)

if Path(LOCAL_DATASET_PATH).exists():
    print(f"Dataset already on local disk: {LOCAL_DATASET_PATH}")
    for split in ['train', 'val', 'test']:
        split_dir = Path(f"{LOCAL_DATASET_PATH}/{split}")
        if split_dir.exists():
            n = len(list(split_dir.glob('*/*_rgb.pt')))
            print(f"   {split.capitalize()} samples: {n}")

elif Path(DRIVE_DATASET_TAR).exists():
    print(f"Found compressed dataset on Drive: {DRIVE_DATASET_TAR}")
    print(f"Copying compressed file to local disk...")

    tar_name = Path(DRIVE_DATASET_TAR).name
    local_tar = f"/dev/shm/{tar_name}"

    !rsync -ah --info=progress2 "{DRIVE_DATASET_TAR}" {local_tar}

    print(f"\nExtracting dataset to local disk...")
    !tar -xzf {local_tar} -C /dev/shm/ 2>&1 | grep -v "Ignoring unknown extended header"

    !rm {local_tar}

    print(f"\nDataset extracted to local disk")

    for split in ['train', 'val', 'test']:
        split_dir = Path(f"{LOCAL_DATASET_PATH}/{split}")
        if split_dir.exists():
            n = len(list(split_dir.glob('*/*_rgb.pt')))
            print(f"   {split.capitalize()} samples: {n}")

else:
    print(f"Dataset not found on Drive!")
    print(f"   Expected location: {DRIVE_DATASET_TAR}")
    raise FileNotFoundError(f"Compressed dataset not found at {DRIVE_DATASET_TAR}")

for meta in ['class_names.txt', 'norm_stats.json']:
    meta_path = Path(LOCAL_DATASET_PATH) / meta
    status = 'OK' if meta_path.exists() else 'MISSING'
    print(f"   {meta}: {status}")

print("\n" + "=" * 60)
print(f"Dataset ready at: {LOCAL_DATASET_PATH}")
print("=" * 60)

## 6. Setup Python Path & Imports

In [ ]:
import sys
import os

modules_to_reload = [k for k in sys.modules.keys() if k.startswith('src.')]
for module in modules_to_reload:
    del sys.modules[module]

project_root = '/content/MSNN-Capstone'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print('Project structure:')
!ls -la {project_root}/src/models/

print('\nImporting dataloaders and ResNet...')
from src.data_utils.washington_dataset import (
    WashingtonRGBDDataset,
    get_washington_dataloaders,
    _load_class_names,
    _load_norm_stats,
)
from src.training.augmentation_config import AugmentationConfig
from src.models.core.resnet import resnet18

print('All imports successful!')

In [ ]:
# Set random seed for reproducibility
from src.utils.seed import set_seed

SEED = 42
DETERMINISTIC = False

set_seed(SEED, deterministic=DETERMINISTIC)

print(f"Seed: {SEED}, Deterministic: {DETERMINISTIC}")

## 7. Configuration

Shared dataset + model-architecture settings, then **one config per baseline** (aug, optimizer, scheduler, training). TODO values should be filled in from HPO results.

In [ ]:
from src.training.augmentation_config import AugmentationConfig

# ======================== DATASET (shared) ========================
DATASET_CONFIG = {
    'data_root': LOCAL_DATASET_PATH,
    'batch_size': 64,
    'num_workers': 5,
    'crop_size': 224,
    'seed': SEED,
}

# ======================== MODEL ARCHITECTURE (shared) ========================
MODEL_CONFIG = {
    'architecture': 'resnet18',
    'num_classes': 51,
    'width_multiplier': 0.75,
    'device': 'cuda',
    'use_amp': True,
}

# ======================== PER-BASELINE CONFIGS ========================
# Each baseline trains independently with its own hyperparameters.
# Fill `aug`, `lr`, `weight_decay`, `dropout_p`, `label_smoothing`, `grad_clip_norm`
# from the corresponding HPO experiment result.
# `early_stopping_patience`: stop if val_mca does not improve for this many consecutive
# epochs; on trigger, best weights are restored before the test eval.

RGB_ONLY_CONFIG = {
    'aug': AugmentationConfig(
        rgb_aug_prob=1.0,       # TODO: fill from HPO
        rgb_aug_mag=1.0,        # TODO: fill from HPO
        depth_aug_prob=1.0,     # unused for RGB-only (no depth stream)
        depth_aug_mag=1.0,      # unused for RGB-only
    ),
    'lr': 1e-4,                 # TODO: fill from HPO
    'weight_decay': 1e-4,       # TODO: fill from HPO
    'dropout_p': 0.4,           # TODO: fill from HPO
    'label_smoothing': 0.1,     # TODO: fill from HPO
    'grad_clip_norm': 1.0,      # TODO: fill from HPO
    'epochs': 150,
    'eta_min': 1e-6,            # TODO: fill from HPO
    'warmup_epochs': 5,
    'warmup_start_factor': 0.2,
    'early_stopping_patience': 20,
}

DEPTH_ONLY_CONFIG = {
    'aug': AugmentationConfig(
        rgb_aug_prob=1.0,       # unused for Depth-only
        rgb_aug_mag=1.0,        # unused for Depth-only
        depth_aug_prob=1.0,     # TODO: fill from HPO
        depth_aug_mag=1.0,      # TODO: fill from HPO
    ),
    'lr': 1e-4,                 # TODO: fill from HPO
    'weight_decay': 1e-4,       # TODO: fill from HPO
    'dropout_p': 0.4,           # TODO: fill from HPO
    'label_smoothing': 0.1,     # TODO: fill from HPO
    'grad_clip_norm': 1.0,      # TODO: fill from HPO
    'epochs': 150,
    'eta_min': 1e-6,            # TODO: fill from HPO
    'warmup_epochs': 5,
    'warmup_start_factor': 0.2,
    'early_stopping_patience': 20,
}

EARLY_FUSION_CONFIG = {
    'aug': AugmentationConfig(
        rgb_aug_prob=1.0,       # TODO: fill from HPO
        rgb_aug_mag=1.0,        # TODO: fill from HPO
        depth_aug_prob=1.0,     # TODO: fill from HPO
        depth_aug_mag=1.0,      # TODO: fill from HPO
    ),
    'lr': 1e-4,                 # TODO: fill from HPO
    'weight_decay': 1e-4,       # TODO: fill from HPO
    'dropout_p': 0.4,           # TODO: fill from HPO
    'label_smoothing': 0.1,     # TODO: fill from HPO
    'grad_clip_norm': 1.0,      # TODO: fill from HPO
    'epochs': 150,
    'eta_min': 1e-6,            # TODO: fill from HPO
    'warmup_epochs': 5,
    'warmup_start_factor': 0.2,
    'early_stopping_patience': 20,
}

LATE_FUSION_CONFIG = {
    'aug': AugmentationConfig(
        rgb_aug_prob=1.0,       # TODO: fill from HPO
        rgb_aug_mag=1.0,        # TODO: fill from HPO
        depth_aug_prob=1.0,     # TODO: fill from HPO
        depth_aug_mag=1.0,      # TODO: fill from HPO
    ),
    'lr': 1e-4,                 # TODO: fill from HPO
    'weight_decay': 1e-4,       # TODO: fill from HPO
    'dropout_p': 0.4,           # TODO: fill from HPO
    'label_smoothing': 0.1,     # TODO: fill from HPO
    'grad_clip_norm': 1.0,      # TODO: fill from HPO
    'epochs': 150,
    'eta_min': 1e-6,            # TODO: fill from HPO
    'warmup_epochs': 5,
    'warmup_start_factor': 0.2,
    'early_stopping_patience': 20,
}

print('All configs defined.')
print(f"  Dataset: {DATASET_CONFIG['data_root']}")
print(f"  Model: ResNet18 ({MODEL_CONFIG['width_multiplier']}x width), {MODEL_CONFIG['num_classes']} classes")
for name, cfg in [('RGB-Only', RGB_ONLY_CONFIG), ('Depth-Only', DEPTH_ONLY_CONFIG),
                   ('Early Fusion', EARLY_FUSION_CONFIG), ('Late Fusion', LATE_FUSION_CONFIG)]:
    print(f"  [{name}] lr={cfg['lr']:.2e}, wd={cfg['weight_decay']:.2e}, dropout={cfg['dropout_p']}, "
          f"label_smooth={cfg['label_smoothing']}, clip={cfg['grad_clip_norm']}, epochs={cfg['epochs']}, "
          f"es_patience={cfg['early_stopping_patience']}")

## 8. Load Dataset

In [ ]:
# Verify dataset structure
from pathlib import Path

print("=" * 60)
print("DATASET STRUCTURE VERIFICATION")
print("=" * 60)

dataset_root = Path(LOCAL_DATASET_PATH)

print("\nDirectory structure:")
print(f"  {dataset_root}/")
for split in ['train', 'val', 'test']:
    split_dir = dataset_root / split
    if split_dir.exists():
        n = len(list(split_dir.glob('*/*_rgb.pt')))
        n_classes = len([d for d in split_dir.iterdir() if d.is_dir()])
        print(f"    {split}/ - {n} paired samples across {n_classes} class folders")

class_names_file = dataset_root / 'class_names.txt'
if class_names_file.exists():
    class_names = _load_class_names(LOCAL_DATASET_PATH)
    print(f"\nClasses ({len(class_names)}):")
    for i, name in enumerate(class_names):
        print(f"  {i}: {name}")

print("\n" + "=" * 60)

In [ ]:
# Helper: build train/val/test loaders for a given baseline config.
# Each baseline needs its own loaders because aug params are per-baseline.
def build_loaders(baseline_config):
    """Return (train_loader, val_loader, test_loader) with this baseline's aug."""
    aug = baseline_config['aug']
    train_loader, val_loader, test_loader, _ = get_washington_dataloaders(
        data_root=DATASET_CONFIG['data_root'],
        batch_size=DATASET_CONFIG['batch_size'],
        num_workers=DATASET_CONFIG['num_workers'],
        crop_size=DATASET_CONFIG['crop_size'],
        seed=DATASET_CONFIG['seed'],
        normalize=True,
        balanced_sampling=True,
        **aug.to_dict(),
    )
    return train_loader, val_loader, test_loader

# Sanity-check: load once with RGB-Only aug to confirm pipeline works and report counts.
print("=" * 60)
print("LOADING WASHINGTON RGB-D DATASET (sanity check)")
print("=" * 60)

_train, _val, _test = build_loaders(RGB_ONLY_CONFIG)

norm_stats = _load_norm_stats(LOCAL_DATASET_PATH)

print(f"\n  Train: {len(_train.dataset):,} samples ({len(_train)} batches)")
print(f"  Val:   {len(_val.dataset):,} samples ({len(_val)} batches)")
print(f"  Test:  {len(_test.dataset):,} samples ({len(_test)} batches)")
print(f"  Norm stats: {list(norm_stats.keys())}")

del _train, _val, _test
print("\n" + "=" * 60)

## 8b. Baselines: ResNet18 (0.75x width)

**Four baselines:**
1. **RGB-only:** Standard ResNet18 with 3-channel RGB input
2. **Depth-only:** ResNet18 with 1-channel depth input
3. **Early fusion:** Concatenate RGB (3ch) + Depth (1ch) = 4-channel input into a single ResNet18
4. **Late fusion:** Two separate ResNet18 backbones (RGB, Depth), features concatenated before a shared classifier

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from src.models.core.resnet import resnet18

# --- Wrapper datasets (preserves dynamic augmentation from underlying dataset) ---
class RGBOnlyDataset(Dataset):
    """Wraps an (rgb, depth, label) dataset to return (rgb, label)."""
    def __init__(self, dataset):
        self.dataset = dataset
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, idx):
        rgb, depth, label = self.dataset[idx]
        return rgb, label

class DepthOnlyDataset(Dataset):
    """Wraps an (rgb, depth, label) dataset to return (depth, label)."""
    def __init__(self, dataset):
        self.dataset = dataset
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, idx):
        rgb, depth, label = self.dataset[idx]
        return depth, label

class EarlyFusionDataset(Dataset):
    """Wraps an (rgb, depth, label) dataset to return (cat(rgb, depth), label)."""
    def __init__(self, dataset):
        self.dataset = dataset
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, idx):
        rgb, depth, label = self.dataset[idx]
        return torch.cat([rgb, depth], dim=0), label


def _eval(model, loader, criterion, device, use_amp, num_classes, is_late_fusion):
    """Evaluate model on loader. Returns (loss, acc, mca)."""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        if is_late_fusion:
            for rgb, depth, labels in loader:
                rgb = rgb.to(device, non_blocking=True)
                depth = depth.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                with torch.amp.autocast('cuda', enabled=use_amp):
                    logits = model(rgb, depth)
                    loss = criterion(logits, labels)
                total_loss += loss.item() * labels.size(0)
                preds = logits.argmax(1)
                correct += preds.eq(labels).sum().item()
                total += labels.size(0)
                all_preds.append(preds.cpu())
                all_labels.append(labels.cpu())
        else:
            for inputs, labels in loader:
                inputs = inputs.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                with torch.amp.autocast('cuda', enabled=use_amp):
                    logits = model(inputs)
                    loss = criterion(logits, labels)
                total_loss += loss.item() * labels.size(0)
                preds = logits.argmax(1)
                correct += preds.eq(labels).sum().item()
                total += labels.size(0)
                all_preds.append(preds.cpu())
                all_labels.append(labels.cpu())
    loss_val = total_loss / total
    acc = correct / total
    all_preds = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)
    per_class = [(all_preds[all_labels == c] == c).float().mean().item()
                 for c in range(num_classes) if (all_labels == c).sum() > 0]
    mca = sum(per_class) / len(per_class)
    return loss_val, acc, mca


def _snapshot_state(model):
    """CPU clone of model.state_dict() for restoring best weights later."""
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def train_baseline(model, train_dl, val_dl, test_dl, config, tag='', is_late_fusion=False):
    """Train a baseline with its own per-model config, monitor val each epoch.

    Best val_mca is tracked and weights are snapshotted on every improvement.
    If val_mca does not improve for `early_stopping_patience` consecutive epochs,
    training stops early and the best weights are restored before the test eval.
    If the schedule completes without triggering, the final-epoch weights are used.
    """
    device = MODEL_CONFIG['device']
    use_amp = MODEL_CONFIG['use_amp']
    num_classes = MODEL_CONFIG['num_classes']
    epochs = config['epochs']
    patience = config.get('early_stopping_patience', None)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config['lr'],
        weight_decay=config['weight_decay'],
    )
    warmup = LinearLR(
        optimizer,
        start_factor=config['warmup_start_factor'],
        end_factor=1.0,
        total_iters=config['warmup_epochs'],
    )
    cosine = CosineAnnealingLR(
        optimizer,
        T_max=epochs - config['warmup_epochs'],
        eta_min=config['eta_min'],
    )
    scheduler = SequentialLR(
        optimizer,
        schedulers=[warmup, cosine],
        milestones=[config['warmup_epochs']],
    )

    criterion = nn.CrossEntropyLoss(label_smoothing=config['label_smoothing'])
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': [], 'val_mca': [],
    }

    best_val_mca = float('-inf')
    best_epoch = 0
    best_state = None
    epochs_since_improve = 0
    early_stopped = False

    for epoch in range(epochs):
        # ------ TRAIN ------
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        if is_late_fusion:
            for rgb, depth, labels in train_dl:
                rgb = rgb.to(device, non_blocking=True)
                depth = depth.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                optimizer.zero_grad()
                with torch.amp.autocast('cuda', enabled=use_amp):
                    logits = model(rgb, depth)
                    loss = criterion(logits, labels)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), config['grad_clip_norm'])
                scaler.step(optimizer)
                scaler.update()
                total_loss += loss.item() * labels.size(0)
                correct += logits.argmax(1).eq(labels).sum().item()
                total += labels.size(0)
        else:
            for inputs, labels in train_dl:
                inputs = inputs.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                optimizer.zero_grad()
                with torch.amp.autocast('cuda', enabled=use_amp):
                    logits = model(inputs)
                    loss = criterion(logits, labels)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), config['grad_clip_norm'])
                scaler.step(optimizer)
                scaler.update()
                total_loss += loss.item() * labels.size(0)
                correct += logits.argmax(1).eq(labels).sum().item()
                total += labels.size(0)

        scheduler.step()
        train_loss = total_loss / total
        train_acc = correct / total
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)

        # ------ VAL ------
        val_loss, val_acc, val_mca = _eval(
            model, val_dl, criterion, device, use_amp, num_classes, is_late_fusion
        )
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_mca'].append(val_mca)

        if val_mca > best_val_mca:
            best_val_mca = val_mca
            best_epoch = epoch + 1
            best_state = _snapshot_state(model)
            epochs_since_improve = 0
        else:
            epochs_since_improve += 1

        if (epoch + 1) % 10 == 0 or epoch == 0:
            lr_now = optimizer.param_groups[0]['lr']
            patience_msg = f' es={epochs_since_improve}/{patience}' if patience is not None else ''
            print(f'  [{tag}] E{epoch+1:3d}/{epochs} | '
                  f'train_loss={train_loss:.4f} train_acc={train_acc:.4f} | '
                  f'val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_mca={val_mca:.4f} | '
                  f'lr={lr_now:.2e}{patience_msg}')

        if patience is not None and epochs_since_improve >= patience:
            early_stopped = True
            print(f"  [{tag}] Early stopping at epoch {epoch+1}: "
                  f"val_mca did not improve for {patience} epochs "
                  f"(best={best_val_mca:.4f} @ epoch {best_epoch})")
            break

    # ------ RESTORE BEST WEIGHTS IF EARLY STOPPED ------
    epochs_completed = len(history['train_loss'])
    if early_stopped and best_state is not None:
        model.load_state_dict(best_state)
        weights_used = f'best @ epoch {best_epoch}'
        print(f"  [{tag}] Restored best weights from epoch {best_epoch} "
              f"(val_mca={best_val_mca:.4f}) for test eval")
    else:
        weights_used = f'final @ epoch {epochs_completed}'

    # ------ FINAL TEST ------
    test_loss, test_acc, test_mca = _eval(
        model, test_dl, criterion, device, use_amp, num_classes, is_late_fusion
    )
    history['test_loss'] = test_loss
    history['test_acc'] = test_acc
    history['test_mca'] = test_mca
    history['best_val_mca'] = best_val_mca
    history['best_val_epoch'] = best_epoch
    history['early_stopped'] = early_stopped
    history['epochs_completed'] = epochs_completed
    history['weights_used'] = weights_used

    print(f"  [{tag}] FINAL [weights={weights_used}] "
          f"test_loss={test_loss:.4f} test_acc={test_acc:.4f} test_mca={test_mca:.4f} "
          f"(best val_mca={best_val_mca:.4f} @ epoch {best_epoch}, early_stopped={early_stopped})")
    return history


print('Wrappers and training helpers defined.')

In [ ]:
# --- RGB-Only: standard 3-channel ResNet18 ---
print('=' * 60)
print(f"BASELINE: ResNet18 {MODEL_CONFIG['width_multiplier']}x width - RGB Only")
print('=' * 60)

from thop import profile

set_seed(SEED, deterministic=DETERMINISTIC)

rgb_only = resnet18(
    num_classes=MODEL_CONFIG['num_classes'],
    width_multiplier=MODEL_CONFIG['width_multiplier'],
    device='cpu',
    use_amp=False,
)

rgb_only.fc = nn.Sequential(
    nn.Dropout(p=RGB_ONLY_CONFIG['dropout_p']),
    nn.Linear(rgb_only.fc.in_features, rgb_only.fc.out_features),
)
rgb_only.to(MODEL_CONFIG['device'])

n_params = sum(p.numel() for p in rgb_only.parameters())
print(f'Parameters: {n_params:,}')

dummy = torch.randn(1, 3, DATASET_CONFIG['crop_size'], DATASET_CONFIG['crop_size']).to(MODEL_CONFIG['device'])
flops, _ = profile(rgb_only, inputs=(dummy,), verbose=False)
print(f'GFLOPs: {flops / 1e9:.3f}')
del dummy

# Build loaders with this baseline's augmentation
rgb_train_base, rgb_val_base, rgb_test_base = build_loaders(RGB_ONLY_CONFIG)
bs = rgb_train_base.batch_size
nw = rgb_train_base.num_workers

rgb_train_dl = DataLoader(RGBOnlyDataset(rgb_train_base.dataset),
                          batch_size=bs, sampler=rgb_train_base.sampler,
                          num_workers=nw, pin_memory=True,
                          persistent_workers=nw > 0)
rgb_val_dl = DataLoader(RGBOnlyDataset(rgb_val_base.dataset),
                        batch_size=bs, shuffle=False,
                        num_workers=max(nw // 2, 0), pin_memory=True,
                        persistent_workers=(nw // 2) > 0)
rgb_test_dl = DataLoader(RGBOnlyDataset(rgb_test_base.dataset),
                         batch_size=bs, shuffle=False,
                         num_workers=max(nw // 2, 0), pin_memory=True,
                         persistent_workers=(nw // 2) > 0)

rgb_history = train_baseline(rgb_only, rgb_train_dl, rgb_val_dl, rgb_test_dl,
                              RGB_ONLY_CONFIG, tag='RGB-Only')
print(f"\nRGB-Only test: acc={rgb_history['test_acc']:.4f}  mca={rgb_history['test_mca']:.4f}")

In [ ]:
# Visualize learned conv1 filters - RGB-Only
import math
from torchvision.utils import make_grid
w = rgb_only.conv1.weight.detach().cpu()
nrow = int(math.ceil(math.sqrt(w.shape[0])))
grid = make_grid(w, nrow=nrow, normalize=True, scale_each=True, pad_value=1)
plt.figure(figsize=(8, 8))
plt.imshow(grid.permute(1, 2, 0).numpy())
plt.axis('off')
plt.title('RGB-Only conv1 filters')
plt.show()

In [ ]:
# --- Depth-Only: 1-channel input ResNet18 ---
print('=' * 60)
print(f"BASELINE: ResNet18 {MODEL_CONFIG['width_multiplier']}x width - Depth Only")
print('=' * 60)

set_seed(SEED, deterministic=DETERMINISTIC)

depth_only = resnet18(
    num_classes=MODEL_CONFIG['num_classes'],
    width_multiplier=MODEL_CONFIG['width_multiplier'],
    device='cpu',
    use_amp=False,
)

# Replace conv1: 1 input channel for depth
old_conv1 = depth_only.conv1
depth_only.conv1 = nn.Conv2d(
    1, old_conv1.out_channels,
    kernel_size=7, stride=2, padding=3, bias=False,
)
nn.init.kaiming_normal_(depth_only.conv1.weight, mode='fan_out', nonlinearity='relu')

depth_only.fc = nn.Sequential(
    nn.Dropout(p=DEPTH_ONLY_CONFIG['dropout_p']),
    nn.Linear(depth_only.fc.in_features, depth_only.fc.out_features),
)
depth_only.to(MODEL_CONFIG['device'])

n_params = sum(p.numel() for p in depth_only.parameters())
print(f'Parameters: {n_params:,}')

dummy = torch.randn(1, 1, DATASET_CONFIG['crop_size'], DATASET_CONFIG['crop_size']).to(MODEL_CONFIG['device'])
flops, _ = profile(depth_only, inputs=(dummy,), verbose=False)
print(f'GFLOPs: {flops / 1e9:.3f}')
del dummy

depth_train_base, depth_val_base, depth_test_base = build_loaders(DEPTH_ONLY_CONFIG)
bs = depth_train_base.batch_size
nw = depth_train_base.num_workers

depth_train_dl = DataLoader(DepthOnlyDataset(depth_train_base.dataset),
                            batch_size=bs, sampler=depth_train_base.sampler,
                            num_workers=nw, pin_memory=True,
                            persistent_workers=nw > 0)
depth_val_dl = DataLoader(DepthOnlyDataset(depth_val_base.dataset),
                          batch_size=bs, shuffle=False,
                          num_workers=max(nw // 2, 0), pin_memory=True,
                          persistent_workers=(nw // 2) > 0)
depth_test_dl = DataLoader(DepthOnlyDataset(depth_test_base.dataset),
                           batch_size=bs, shuffle=False,
                           num_workers=max(nw // 2, 0), pin_memory=True,
                           persistent_workers=(nw // 2) > 0)

depth_history = train_baseline(depth_only, depth_train_dl, depth_val_dl, depth_test_dl,
                                DEPTH_ONLY_CONFIG, tag='Depth-Only')
print(f"\nDepth-Only test: acc={depth_history['test_acc']:.4f}  mca={depth_history['test_mca']:.4f}")

In [ ]:
# Visualize learned conv1 filters - Depth-Only (1-channel -> repeat to 3 for display)
import math
from torchvision.utils import make_grid
w = depth_only.conv1.weight.detach().cpu()
n_filters, in_ch, kh, kw = w.shape
w = w.repeat(1, 3, 1, 1)
nrow = int(math.ceil(math.sqrt(n_filters)))
grid = make_grid(w, nrow=nrow, normalize=True, scale_each=True, pad_value=1)
plt.figure(figsize=(8, 8))
plt.imshow(grid.permute(1, 2, 0).numpy())
plt.axis('off')
plt.title('Depth-Only conv1 filters')
plt.show()

In [ ]:
# --- Early Fusion: 4-channel input ResNet18 ---
print('=' * 60)
print(f"BASELINE: ResNet18 {MODEL_CONFIG['width_multiplier']}x width - Early Fusion (RGB+Depth concat)")
print('=' * 60)

set_seed(SEED, deterministic=DETERMINISTIC)

early_fusion = resnet18(
    num_classes=MODEL_CONFIG['num_classes'],
    width_multiplier=MODEL_CONFIG['width_multiplier'],
    device='cpu',
    use_amp=False,
)

# Replace conv1: 4 input channels (RGB=3 + Depth=1)
old_conv1 = early_fusion.conv1
early_fusion.conv1 = nn.Conv2d(
    4, old_conv1.out_channels,
    kernel_size=7, stride=2, padding=3, bias=False,
)
nn.init.kaiming_normal_(early_fusion.conv1.weight, mode='fan_out', nonlinearity='relu')

early_fusion.fc = nn.Sequential(
    nn.Dropout(p=EARLY_FUSION_CONFIG['dropout_p']),
    nn.Linear(early_fusion.fc.in_features, early_fusion.fc.out_features),
)
early_fusion.to(MODEL_CONFIG['device'])

n_params = sum(p.numel() for p in early_fusion.parameters())
print(f'Parameters: {n_params:,}')

dummy = torch.randn(1, 4, DATASET_CONFIG['crop_size'], DATASET_CONFIG['crop_size']).to(MODEL_CONFIG['device'])
flops, _ = profile(early_fusion, inputs=(dummy,), verbose=False)
print(f'GFLOPs: {flops / 1e9:.3f}')
del dummy

ef_train_base, ef_val_base, ef_test_base = build_loaders(EARLY_FUSION_CONFIG)
bs = ef_train_base.batch_size
nw = ef_train_base.num_workers

ef_train_dl = DataLoader(EarlyFusionDataset(ef_train_base.dataset),
                         batch_size=bs, sampler=ef_train_base.sampler,
                         num_workers=nw, pin_memory=True,
                         persistent_workers=nw > 0)
ef_val_dl = DataLoader(EarlyFusionDataset(ef_val_base.dataset),
                       batch_size=bs, shuffle=False,
                       num_workers=max(nw // 2, 0), pin_memory=True,
                       persistent_workers=(nw // 2) > 0)
ef_test_dl = DataLoader(EarlyFusionDataset(ef_test_base.dataset),
                        batch_size=bs, shuffle=False,
                        num_workers=max(nw // 2, 0), pin_memory=True,
                        persistent_workers=(nw // 2) > 0)

ef_history = train_baseline(early_fusion, ef_train_dl, ef_val_dl, ef_test_dl,
                             EARLY_FUSION_CONFIG, tag='EarlyFusion')
print(f"\nEarly Fusion test: acc={ef_history['test_acc']:.4f}  mca={ef_history['test_mca']:.4f}")

In [ ]:
# Visualize learned conv1 filters - Early Fusion (4ch: RGB+Depth)
import math
from torchvision.utils import make_grid
w = early_fusion.conv1.weight.detach().cpu()
nrow = int(math.ceil(math.sqrt(w.shape[0])))
fig, axes = plt.subplots(2, 1, figsize=(8, 16))

rgb_filters = w[:, :3]
rgb_grid = make_grid(rgb_filters, nrow=nrow, normalize=True, scale_each=True, pad_value=1)
axes[0].imshow(rgb_grid.permute(1, 2, 0).numpy())
axes[0].axis('off')
axes[0].set_title('Early Fusion conv1 filters - RGB channels (first 3)')

depth_filters = w[:, 3:4].repeat(1, 3, 1, 1)
depth_grid = make_grid(depth_filters, nrow=nrow, normalize=True, scale_each=True, pad_value=1)
axes[1].imshow(depth_grid.permute(1, 2, 0).numpy())
axes[1].axis('off')
axes[1].set_title('Early Fusion conv1 filters - Depth channel (repeated to RGB)')
plt.tight_layout()
plt.show()

In [ ]:
# --- Late Fusion: two ResNet18 backbones + shared head ---
print('=' * 60)
print(f"BASELINE: ResNet18 {MODEL_CONFIG['width_multiplier']}x width - Late Fusion (feature concat)")
print('=' * 60)

class LateFusionResNet(nn.Module):
    def __init__(self, num_classes, width_multiplier, dropout_p):
        super().__init__()
        self.rgb_backbone = resnet18(
            num_classes=num_classes,
            width_multiplier=width_multiplier,
            device='cpu', use_amp=False,
        )
        self.depth_backbone = resnet18(
            num_classes=num_classes,
            width_multiplier=width_multiplier,
            device='cpu', use_amp=False,
        )
        self.depth_backbone.conv1 = nn.Conv2d(
            1, self.depth_backbone.conv1.out_channels,
            kernel_size=7, stride=2, padding=3, bias=False,
        )
        nn.init.kaiming_normal_(self.depth_backbone.conv1.weight, mode='fan_out', nonlinearity='relu')

        feat_dim = self.rgb_backbone.fc.in_features
        self.rgb_backbone.fc = nn.Identity()
        self.depth_backbone.fc = nn.Identity()

        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout_p),
            nn.Linear(feat_dim * 2, num_classes),
        )

    def forward(self, rgb, depth):
        rgb_feat = self.rgb_backbone(rgb)
        depth_feat = self.depth_backbone(depth)
        fused = torch.cat([rgb_feat, depth_feat], dim=1)
        return self.classifier(fused)


set_seed(SEED, deterministic=DETERMINISTIC)

late_fusion = LateFusionResNet(
    num_classes=MODEL_CONFIG['num_classes'],
    width_multiplier=MODEL_CONFIG['width_multiplier'],
    dropout_p=LATE_FUSION_CONFIG['dropout_p'],
).to(MODEL_CONFIG['device'])

n_params = sum(p.numel() for p in late_fusion.parameters())
print(f'Parameters: {n_params:,}')

dummy_rgb = torch.randn(1, 3, DATASET_CONFIG['crop_size'], DATASET_CONFIG['crop_size']).to(MODEL_CONFIG['device'])
dummy_depth = torch.randn(1, 1, DATASET_CONFIG['crop_size'], DATASET_CONFIG['crop_size']).to(MODEL_CONFIG['device'])
lf_flops, _ = profile(late_fusion, inputs=(dummy_rgb, dummy_depth), verbose=False)
print(f'GFLOPs: {lf_flops / 1e9:.3f}')
del dummy_rgb, dummy_depth

# Late fusion uses the raw (rgb, depth, label) loaders - no wrapper needed.
lf_train_dl, lf_val_dl, lf_test_dl = build_loaders(LATE_FUSION_CONFIG)

lf_history = train_baseline(late_fusion, lf_train_dl, lf_val_dl, lf_test_dl,
                             LATE_FUSION_CONFIG, tag='LateFusion', is_late_fusion=True)
print(f"\nLate Fusion test: acc={lf_history['test_acc']:.4f}  mca={lf_history['test_mca']:.4f}")

In [ ]:
# Visualize learned conv1 filters - Late Fusion (RGB + Depth backbones)
import math
from torchvision.utils import make_grid
fig, axes = plt.subplots(2, 1, figsize=(8, 16))
for si, (backbone, label) in enumerate([
    (late_fusion.rgb_backbone, 'RGB Backbone'),
    (late_fusion.depth_backbone, 'Depth Backbone'),
]):
    w = backbone.conv1.weight.detach().cpu()
    if w.shape[1] == 1:
        w = w.repeat(1, 3, 1, 1)
    nrow = int(math.ceil(math.sqrt(w.shape[0])))
    grid = make_grid(w, nrow=nrow, normalize=True, scale_each=True, pad_value=1)
    axes[si].imshow(grid.permute(1, 2, 0).numpy())
    axes[si].axis('off')
    axes[si].set_title(f'Late Fusion conv1 filters - {label}')
plt.tight_layout()
plt.show()

## 9. Training Curves

In [ ]:
# --- Training curves: all four baselines ---
# Each baseline may have a different `epochs` count, so x-axis is per-baseline.
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

colors = {'RGB-Only': 'tab:blue', 'Depth-Only': 'tab:orange',
          'Early Fusion': 'tab:green', 'Late Fusion': 'tab:purple'}
histories = {
    'RGB-Only': rgb_history,
    'Depth-Only': depth_history,
    'Early Fusion': ef_history,
    'Late Fusion': lf_history,
}

for name, h in histories.items():
    c = colors[name]
    ep = range(1, len(h['train_loss']) + 1)
    axes[0, 0].plot(ep, h['train_loss'], label=f'{name} train', color=c)
    axes[0, 0].plot(ep, h['val_loss'], label=f'{name} val', color=c, linestyle='--')
    axes[0, 1].plot(ep, h['train_acc'], label=f'{name} train', color=c)
    axes[0, 1].plot(ep, h['val_acc'], label=f'{name} val', color=c, linestyle='--')
    axes[1, 0].plot(ep, h['val_mca'], label=name, color=c)
    axes[1, 1].axhline(y=h['test_mca'], color=c, linestyle='-', alpha=0.8,
                       label=f"{name} test_mca ({h['test_mca']:.3f})")

axes[0, 0].set_xlabel('Epoch'); axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Loss (train solid, val dashed)')
axes[0, 0].legend(fontsize=8); axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].set_xlabel('Epoch'); axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].set_title('Accuracy (train solid, val dashed)')
axes[0, 1].legend(fontsize=8); axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].set_xlabel('Epoch'); axes[1, 0].set_ylabel('Val MCA')
axes[1, 0].set_title('Val Mean Class Accuracy')
axes[1, 0].legend(fontsize=9); axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].set_xlabel(''); axes[1, 1].set_ylabel('Test MCA')
axes[1, 1].set_title('Final Test MCA')
axes[1, 1].legend(fontsize=9); axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_xlim(0, 1)

plt.suptitle(f"ResNet18 ({MODEL_CONFIG['width_multiplier']}x) Baselines on Washington RGB-D Object", fontsize=13)
plt.tight_layout()
plt.show()

## 10. Baseline Results Summary

In [ ]:
# --- Summary table ---
print('=' * 115)
print(f"BASELINE RESULTS - ResNet18 ({MODEL_CONFIG['width_multiplier']}x width) on Washington RGB-D Object 51-Category")
print('=' * 115)
print(f'{"Model":<14} {"Test Acc":>9} {"Test MCA":>9} {"Test Loss":>10} {"Best Val MCA":>13} {"Best Epoch":>11} {"Epochs Run":>11} {"Early Stop":>11} {"Weights":>22}')
print('-' * 115)

results = {
    'RGB-Only': rgb_history,
    'Depth-Only': depth_history,
    'Early Fusion': ef_history,
    'Late Fusion': lf_history,
}

for name, h in results.items():
    weights_label = h.get('weights_used', 'final')
    epochs_run = h.get('epochs_completed', len(h['train_loss']))
    es_flag = 'yes' if h.get('early_stopped', False) else 'no'
    print(f"{name:<14} {h['test_acc']:>9.4f} {h['test_mca']:>9.4f} {h['test_loss']:>10.4f} "
          f"{h['best_val_mca']:>13.4f} {h['best_val_epoch']:>11} {epochs_run:>11} {es_flag:>11} {weights_label:>22}")

print('=' * 115)

# Bar chart comparison
fig, ax = plt.subplots(figsize=(9, 5))
names = list(results.keys())
mcas = [results[n]['test_mca'] for n in names]
accs = [results[n]['test_acc'] for n in names]

x = range(len(names))
width = 0.35
ax.bar([i - width/2 for i in x], accs, width, label='Test Accuracy', color='steelblue')
ax.bar([i + width/2 for i in x], mcas, width, label='Mean Class Accuracy', color='coral')
ax.set_xticks(list(x))
ax.set_xticklabels(names)
ax.set_ylabel('Score')
ax.set_title('Baseline Comparison - Washington RGB-D Object 51-Category')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 1)

for i, (a, m) in enumerate(zip(accs, mcas)):
    ax.text(i - width/2, a + 0.01, f'{a:.3f}', ha='center', fontsize=9)
    ax.text(i + width/2, m + 0.01, f'{m:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()